<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_02_control_panel.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.2 · Notebook 02 — the control panel

**Paired with L9.2 · Turbulent Flow**

Choose the shape, the flow and the numerics with widgets. **Preview geometry**
is free and instant; use it before spending minutes on a run.

Paste your `residual_fn` and `loss_fn_factory` from notebook 01 into the second
cell.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-channel-flow/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
# --- paste your residual_fn and loss_fn_factory from notebook 01 here ---
raise NotImplementedError("paste your residual_fn and loss_fn_factory")

## 1 · The panel

Every run is appended to `cases`, and notebook 04 turns that list into the
report table.

In [ ]:
cases = []

def on_run(cfg):
    r = pb.run_case(cfg, residual_fn, loss_fn_factory)
    cases.append(r)
    plot_curves(r["history"], title=f"{cfg.shape}, Re = {cfg.reynolds:g}")
    plt.show()
    pb.plot_case(r)
    print(f"\n{len(cases)} case(s) recorded")

panel = pb.control_panel(on_run)

## 2 · Suggested studies

| Study | Vary | Hold fixed | Look for |
|---|---|---|---|
| Shape | all five shapes | Re, blockage | which has the highest drag, and where it separates |
| Reynolds | Re 5 → 3000 | shape, blockage | where the solution degrades |
| Blockage | size 0.05 → 0.35 | shape, Re | how pressure drop scales |
| Closure | none vs uniform | everything | what an eddy viscosity actually changes |
| Inlet | uniform vs Poiseuille | everything | whether the entrance region matters |

The readout warns when N_f/P\* drops below 10 or the blockage exceeds 0.5.

---

## 3 · Save

In [ ]:
import pickle

os.makedirs("Ex09.2_outputs", exist_ok=True)
path = os.path.join("Ex09.2_outputs", "nb02_cases.pkl")
with open(path, "wb") as f:
    pickle.dump([{k: v for k, v in c.items() if k != "model"} for c in cases], f)
print(f"wrote {path}  ({len(cases)} cases)")